# test for vorarlberg

In [478]:
import os

import pandas as pd
import numpy as np

import regex as re

In [479]:
data_path = "../data/in/"
output_path = "../data/out/"

regions = {"vor": "20241214-0617_gtfs_vor_2024", #vienna, lower austria, burgenland
           "ooev": "20241212-0156_gtfs_ooevv_2024", #upper austria
           "esg": "20241203-0058_gtfs_esg_2024", #linz
           "verbund": "20241217-0310_gtfs_verbundlinie_2024", #styria
           "kaernter": "20241214-0253_gtfs_kaerntnerlinien_2024", #carinthia
           "salzburg": "20241217-0359_gtfs_salzburgverkehr_2024", #salzburg
           "vvt": "20241217-0436_gtfs_vvt_2024", #tyrol
           "vmobil": "20241212-0624_gtfs_vmobil_2024", #vorarlberg
           "oebb": "GTFS_2024_obb"} #oebb maybe 20241217-0222_gtfs_evu_2024

# select region
state_name = "vmobil"
# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240530

# stop categories
table = np.array([
    ["I", "I", "II", "III"],        # < 5 min
    ["I", "II", "III", "III"],      # 5 >= x <= 10
    ["II", "III", "IV", "IV"],      # 10 < x < 20
    ["III", "IV", "V", "V"],        # 20 >= x < 40
    ["IV", "V", "VI", "VI"],        # 40 >= x <= 60
    ["V", "VI", "VII", "VII"],      # 60 < x <= 120  
    ["", "VII", "VIII", "VIII"],    # 120 < x <= 210 
    ["", "", "", ""],               # > 210

])

transport_category = ["Fernverkehr REX", 
                      "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
                      "Straßenbahn, Metrobus, 0-Bus", 
                      "Bus"]


In [480]:
def lookup_category(interval, t_cat):
    if interval < 5:
        return table[0][t_cat]
    elif interval <= 10:
        return table[1][t_cat]
    elif interval < 20:
        return table[2][t_cat]
    elif interval < 40:
        return table[3][t_cat]
    elif interval <= 60:
        return table[4][t_cat]
    elif interval <= 120:
        return table[5][t_cat]
    elif interval <= 210:
        return table[6][t_cat]
    else:
        return table[7][t_cat]
    
# def lookup_category(interval, t_cat):
#     # thresholds = [5, 10, 20, 40, 60, 120, 210]
#     # adjust threshold to work with '<=', i.e. for '< x' use x - 1
#     thresholds = [4, 10, 19, 39, 60, 120, 210]

#     for i, threshold in enumerate(thresholds):
#         if interval <= threshold:    
#             return table[i][t_cat]
    
#     # return last entry as default
#     return table[-1][t_cat]

In [481]:

path = f"{data_path}/{regions[state_name]}/"

stops_df = pd.read_csv(path + "stops.txt", quotechar='"', sep=",")

display(stops_df.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:47:1222:0:4,St. Anton a.A. Bahnhof,47.127468,10.266639,6455.0,NaN,Pat:47:1222,Level 0,1
1,at:47:1222:22,Steig 2+3,47.127450,10.267232,NaN,NaN,Pat:47:1222,Level 0,NaN
2,at:47:61099:0:1,St. Anton a. A. Kohlereck,47.122358,10.253820,6455.0,NaN,Pat:47:61099,Level 0,1
3,at:47:61099:0:2,St. Anton a. A. Kohlereck,47.122322,10.253703,6455.0,NaN,Pat:47:61099,Level 0,2
4,at:47:62209:0:1,St. Anton a. A. Stadle B197,47.122273,10.248646,6455.0,NaN,Pat:47:62209,Level 0,1


In [482]:
stop_times_df = pd.read_csv(path + "stop_times.txt", sep=',', quotechar='"')

# remove from stop_times_df entries outside of the time window (6am-8pm)
stop_times_df = stop_times_df[stop_times_df['departure_time'].between('06:00:00', '20:00:00')]

display(stop_times_df)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
27,1.T0.12-820-E-j24-1.20.R,06:00:00,06:00:00,at:48:1232:0:2,28,NaN,0,0,19261.74
28,1.T0.12-820-E-j24-1.20.R,06:01:00,06:01:00,at:48:1233:0:2,29,NaN,0,0,20149.77
29,1.T0.12-820-E-j24-1.20.R,06:02:00,06:02:00,at:48:1228:0:2,30,NaN,0,0,20924.20
30,1.T0.12-820-E-j24-1.20.R,06:03:00,06:03:00,at:48:1229:0:2,31,NaN,0,0,21372.79
31,1.T0.12-820-E-j24-1.20.R,06:05:00,06:05:00,at:48:1235:0:2,32,NaN,0,0,22450.46
...,...,...,...,...,...,...,...,...,...
388148,99.T3.91-351-E-j24-1.10.H,09:51:00,09:51:00,ch:23016:31934:0:1,15,NaN,0,0,8933.48
388149,99.T3.91-351-E-j24-1.10.H,09:52:00,09:52:00,ch:23016:31936:0:1,16,NaN,0,0,9302.05
388150,99.T3.91-351-E-j24-1.10.H,09:53:00,09:53:00,ch:23016:31935:0:1,17,NaN,0,0,9741.97
388151,99.T3.91-351-E-j24-1.10.H,09:54:00,09:54:00,ch:23016:29838:0:1,18,NaN,0,0,10215.69


In [483]:
trips_df = pd.read_csv(path + 'trips.txt', sep=',', quotechar='"')

display(trips_df.head())

,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
1,at:vvv:101:,T2,1.T2.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
2,at:vvv:101:,T3,1.T3.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
4,at:vvv:101:,T2,10.T2.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


In [484]:
calendar_df = pd.read_csv(path + "calendar.txt", sep=",", quotechar='"')

display(calendar_df.head())

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,T0,1,1,1,1,1,0,0,20231210,20241214
1,T0#1,1,1,1,1,1,0,0,20240708,20240913
2,T0+02o00,1,1,1,1,1,0,0,20231210,20241214
3,T0+05310,0,0,0,0,1,0,0,20231210,20241214
4,T0+05p00,1,1,1,1,1,0,0,20231210,20241214


In [485]:
# remove all services entries in calendar_df that are not relevant for the specified day

calendar_filtered_df = calendar_df[(calendar_df['start_date'] <= selected_day) & (calendar_df['end_date'] >= selected_day)]
# TODO: check exceptions from calendar_dates_df
calendar_filtered_df.head()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,T0,1,1,1,1,1,0,0,20231210,20241214
2,T0+02o00,1,1,1,1,1,0,0,20231210,20241214
3,T0+05310,0,0,0,0,1,0,0,20231210,20241214
4,T0+05p00,1,1,1,1,1,0,0,20231210,20241214
5,T0+09610,1,1,0,0,0,0,0,20231210,20241214


In [486]:
calendar_dates_df = pd.read_csv(path + '/calendar_dates.txt', sep=',', quotechar='"')
print(calendar_dates_df.shape)
calendar_dates_df.head()

(86210, 3)


,service_id,date,exception_type
0,T0,20231225,2
1,T0,20231226,2
2,T0,20240101,2
3,T0,20240401,2
4,T0,20240501,2


In [487]:
calendar_dates_filtered = calendar_dates_df[calendar_dates_df["date"] == selected_day]
print(calendar_dates_filtered.shape)
calendar_dates_filtered.head()

(505, 3)


,service_id,date,exception_type
7,T0,20240530,2
48,T0+02o00,20240530,2
250,T0+05p00,20240530,2
503,T0+0a710,20240530,2
591,T0+0to00,20240530,2


In [488]:
# remove entries from trips_df that are not valid i.e only valid ones
trips_filtered_df = trips_df[trips_df['service_id'].isin(calendar_filtered_df['service_id'])]
print(trips_filtered_df.shape)
trips_filtered_df.head()

(19548, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
1,at:vvv:101:,T2,1.T2.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
2,at:vvv:101:,T3,1.T3.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
4,at:vvv:101:,T2,10.T2.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


In [489]:
# remove all services that have exception_type == 2 (removed) in calendar_dates
trips_filtered_df = trips_filtered_df[trips_filtered_df["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"] == 2]["service_id"])]
print(trips_filtered_df.shape)
trips_filtered_df.head()
#len(trips_filtered_df["trip_id"].unique())

(11597, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
5,at:vvv:101:,T0,11.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
8,at:vvv:101:,T0,12.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
10,at:vvv:101:,T0,13.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


In [490]:
# add  back services added in calendar_dates (exception_type=1)
# trips_filtered = trips_filtered + trips[service_id = calender_dates[exception_type=1][service_id]]
trips_full = pd.concat([trips_filtered_df, trips_df[trips_df["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"]==1]["service_id"])]])
print(trips_full.shape)
trips_full.head()
#len(trips_full["trip_id"].unique())

(13325, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
5,at:vvv:101:,T0,11.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
8,at:vvv:101:,T0,12.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
10,at:vvv:101:,T0,13.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


In [491]:
print(stop_times_df.shape)

# keep only valid trips from trips_filtered_df
stop_times_df = stop_times_df[stop_times_df['trip_id'].isin(trips_filtered_df['trip_id'])]

print(stop_times_df.shape)

(347880, 9)
(200136, 9)


In [492]:
df = stops_df.merge(stop_times_df, on='stop_id')
result = df.groupby("parent_station").size().reset_index(name="count")
result.rename(columns={"parent_station": "stop_id"}, inplace=True)
display(result.head())

,stop_id,count
0,Pat:47:1222,71
1,Pat:47:61099,71
2,Pat:47:62209,36
3,Pat:47:62504,36
4,Pat:47:64938,36


In [493]:
# merge result with stops parent stations
stops_final_df = stops_df.merge(result, on='stop_id', how='right')
display(stops_final_df.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code,count
0,Pat:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,NaN,1.0,NaN,NaN,NaN,71
1,Pat:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,NaN,1.0,NaN,NaN,NaN,71
2,Pat:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,NaN,1.0,NaN,NaN,NaN,36
3,Pat:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,NaN,1.0,NaN,NaN,NaN,36
4,Pat:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,NaN,1.0,NaN,NaN,NaN,36


In [494]:
# calculate the interval

# stops_final_df["interval"] = stops_final_df["count"].apply(lambda x: 840 / (x/2))
# display(stops_final_df.sort_values(by="interval", ascending=True))

stop_count = stops_final_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'count']]

stop_count['stop_id'] = stop_count['stop_id'].str[1:]
stop_count = stop_count[stop_count['stop_id'].str.startswith('at')]

stop_count

/var/folders/t9/cqlkdlt50h94khlrvmtm0zbc0000gn/T/ipykernel_14095/1283220381.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stop_count['stop_id'] = stop_count['stop_id'].str[1:]


,stop_id,stop_name,stop_lat,stop_lon,count
0,at:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,71
1,at:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,71
2,at:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,36
3,at:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,36
4,at:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,36
...,...,...,...,...,...
1702,at:48:992,Gisingen Ketschelenstraße,47.259808,9.590279,122
1703,at:48:993,Gisingen Lehrer-Frick-Straße,47.261467,9.592058,57
1704,at:48:995,Gisingen Milchhof,47.254772,9.583955,79
1705,at:48:996,Gisingen Oberaustraße,47.252907,9.587324,72


In [495]:
stop_count.to_csv(output_path + f'stop_count_{state_name}.csv')

In [496]:
# Betrachtungszeitraum: 6–20 Uhr (= 840 Minuten)
# Stichtage: Werktag ohne Schule (Herbstferien): im Jahr 2021 der 28. 10.
# Normaler Werktag mit Schule: im Jahr 2021 der 22. 10.
# Intervallberechnung: Bildung der Summe der
# Abfahrten aller Verkehrsmittel über alle Ver-
# kehrsmittelkategorien, Multiplikation mit einem
# Richtungsfaktor von 0,5 und Berechnung des
# durchschnittlichen Intervalls über den gesamten
# Betrachtungszeitraum pro Richtung (840 Minuten
# dividiert durch die Zahl der Abfahrten pro Rich-
# tung). Der Richtungsfaktor wird auf allen Linien
# angewendet, Rundlinien ebenfalls.

# idea:
# 0. select specific day
# 1. take trip_id from entry in stop_times_df
# 2. look for that trip_id in trips_df
# 3. check if service_id from corresponding trip entry in calender_df is in valid time range,
#    also check exceptions in calender_date_df
# 4. if stop is valid, add to data for stop_id parent station


In [497]:
# code to determine transport categories
# transport_category = ["Fernverkehr REX", 
                    #   "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
                    #   "Straßenbahn, Metrobus, 0-Bus", 
                    #   "Bus"]

# {0: "Tram", 1: "Metro", 2: "Fernverkehr REX", 3: "Bus", 11: "Electric Bus", }
route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3, }

In [498]:
routes_df = pd.read_csv(path + 'routes.txt', sep=',', quotechar='"')
display(routes_df[["route_id", "route_type"]].head())

,route_id,route_type
0,at:vvv:101:,3
1,at:vvv:102:,3
2,at:vvv:103:,3
3,at:vvv:104:,3
4,at:vvv:105:,3


In [499]:
display(trips_filtered_df.head())
stop_type_df = trips_filtered_df[["route_id", "trip_id"]].merge(routes_df[["route_id", "route_type"]], on='route_id', how='left')
display(stop_type_df.head())

,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
5,at:vvv:101:,T0,11.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
8,at:vvv:101:,T0,12.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
10,at:vvv:101:,T0,13.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


,route_id,trip_id,route_type
0,at:vvv:101:,1.T0.31-101-E-j24-1.1.H,3
1,at:vvv:101:,10.T0.31-101-E-j24-1.2.H,3
2,at:vvv:101:,11.T0.31-101-E-j24-1.2.H,3
3,at:vvv:101:,12.T0.31-101-E-j24-1.2.H,3
4,at:vvv:101:,13.T0.31-101-E-j24-1.2.H,3


In [500]:
display(stop_times_df.head())
stop_type_df = stop_type_df[["trip_id", "route_type"]].merge(stop_times_df[['stop_id', 'trip_id']], on='trip_id', how='left')
display(stop_type_df.head())

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
27,1.T0.12-820-E-j24-1.20.R,06:00:00,06:00:00,at:48:1232:0:2,28,NaN,0,0,19261.74
28,1.T0.12-820-E-j24-1.20.R,06:01:00,06:01:00,at:48:1233:0:2,29,NaN,0,0,20149.77
29,1.T0.12-820-E-j24-1.20.R,06:02:00,06:02:00,at:48:1228:0:2,30,NaN,0,0,20924.20
30,1.T0.12-820-E-j24-1.20.R,06:03:00,06:03:00,at:48:1229:0:2,31,NaN,0,0,21372.79
31,1.T0.12-820-E-j24-1.20.R,06:05:00,06:05:00,at:48:1235:0:2,32,NaN,0,0,22450.46


,trip_id,route_type,stop_id
0,1.T0.31-101-E-j24-1.1.H,3,at:48:452:0:5
1,1.T0.31-101-E-j24-1.1.H,3,at:48:469:0:2
2,1.T0.31-101-E-j24-1.1.H,3,at:48:466:0:2
3,1.T0.31-101-E-j24-1.1.H,3,at:48:471:0:2
4,1.T0.31-101-E-j24-1.1.H,3,at:48:501:0:1


In [501]:
stops_df.head()
stop_type_df = stop_type_df[["stop_id", "route_type"]].merge(stops_df[["stop_id", "parent_station"]], on="stop_id", how="left")
display(stop_type_df.head())
print(stop_type_df.shape)

,stop_id,route_type,parent_station
0,at:48:452:0:5,3,Pat:48:452
1,at:48:469:0:2,3,Pat:48:469
2,at:48:466:0:2,3,Pat:48:466
3,at:48:471:0:2,3,Pat:48:471
4,at:48:501:0:1,3,Pat:48:501


(200914, 3)


In [502]:
station_type_df = stop_type_df.copy()
station_type_df["rank"] = station_type_df["route_type"].apply(lambda x: route_type_translation[x]).min()
station_type_df = station_type_df.groupby("parent_station")["rank"].min().reset_index()
display(station_type_df.head())
print(station_type_df.shape)

,parent_station,rank
0,Pat:47:1222,3
1,Pat:47:61099,3
2,Pat:47:62209,3
3,Pat:47:62504,3
4,Pat:47:64938,3


(1965, 2)


In [503]:
station_type_df.rename(columns={'parent_station':'stop_id'}, inplace=True)
stops_final_df = stops_final_df.merge(station_type_df, on='stop_id', how='left')
display(stops_final_df.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code,count,rank
0,Pat:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,NaN,1.0,NaN,NaN,NaN,71,3
1,Pat:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,NaN,1.0,NaN,NaN,NaN,71,3
2,Pat:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,NaN,1.0,NaN,NaN,NaN,36,3
3,Pat:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,NaN,1.0,NaN,NaN,NaN,36,3
4,Pat:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,NaN,1.0,NaN,NaN,NaN,36,3


In [504]:
export = stops_final_df.drop(columns=["platform_code", "parent_station", "zone_id", "location_type", "level_id"], axis=1)
export

,stop_id,stop_name,stop_lat,stop_lon,count,rank
0,Pat:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,71,3
1,Pat:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,71,3
2,Pat:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,36,3
3,Pat:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,36,3
4,Pat:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,36,3
...,...,...,...,...,...,...
1960,Pfl:21:810,Eschen Sportpark,47.205667,9.533982,130,3
1961,Pfl:21:814,Eschen Kohlplatz,47.211068,9.528322,74,3
1962,Pfl:21:841,Eschen Presta,47.207565,9.528179,132,3
1963,Pfl:21:911,Schaan Theater,47.168200,9.512027,67,3


In [505]:
export['stop_id'] = export['stop_id'].str[1:]
export = export[export['stop_id'].str.startswith('at')]
export

,stop_id,stop_name,stop_lat,stop_lon,count,rank
0,at:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,71,3
1,at:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,71,3
2,at:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,36,3
3,at:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,36,3
4,at:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,36,3
...,...,...,...,...,...,...
1702,at:48:992,Gisingen Ketschelenstraße,47.259808,9.590279,122,3
1703,at:48:993,Gisingen Lehrer-Frick-Straße,47.261467,9.592058,57,3
1704,at:48:995,Gisingen Milchhof,47.254772,9.583955,79,3
1705,at:48:996,Gisingen Oberaustraße,47.252907,9.587324,72,3


In [506]:
# export["rank"] = export.apply(lambda x: lookup_category(x["interval"], x["category"]), axis=1)

In [507]:
export.to_csv(output_path + f"stop_rank_{state_name}.csv", index=False)

# Adding OEBB data

In [508]:
obb_path = data_path + "/GTFS_2024_obb"

stops_df_obb = pd.read_csv(obb_path + "/stops.txt", quotechar='"', sep=",")
stop_times_df_obb = pd.read_csv(obb_path + "/stop_times.txt", quotechar='"', sep=",")
trips_df_obb = pd.read_csv(obb_path + "/trips.txt", quotechar='"', sep=",")
routes_df_obb = pd.read_csv(obb_path + "/routes.txt", quotechar='"', sep=",")
calendar_df_obb = pd.read_csv(obb_path + "/calendar.txt", quotechar='"', sep=",")
calendar_dates_df_obb = pd.read_csv(obb_path + "/calendar_dates.txt", quotechar='"', sep=",")

print(trips_df_obb.shape)
trips_df_obb.head()

(20443, 8)


/var/folders/t9/cqlkdlt50h94khlrvmtm0zbc0000gn/T/ipykernel_14095/497397612.py:4: DtypeWarning: Columns (5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times_df_obb = pd.read_csv(obb_path + "/stop_times.txt", quotechar='"', sep=",")


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,10-A10-j24-1,TA+a3,1.TA.10-A10-j24-1.1.R,10-A10-j24-1.1.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
1,10-A10-j24-1,TA+a2300,2.TA.10-A10-j24-1.2.R,10-A10-j24-1.2.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
2,10-A10-j24-1,TA+o6300,3.TA.10-A10-j24-1.3.R,10-A10-j24-1.3.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
3,10-A10-j24-1,TA+qf300,4.TA.10-A10-j24-1.4.R,10-A10-j24-1.4.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
4,10-A10-j24-1,TA+vW,5.TA.10-A10-j24-1.5.R,10-A10-j24-1.5.R,Wien Hauptbahnhof,D 14317,1.0,NaN


In [509]:
stop_df_obb_mod = stops_df_obb.copy()
stop_df_obb_mod['stop_id'] = stop_df_obb_mod['stop_id'].astype(str)
stop_df_obb_mod["stop_id"] = stop_df_obb_mod["stop_id"].str.extract(r'^((?:[^:]*:){3})')[0].str.rstrip(':')
stop_df_obb_mod.sort_values(by=['parent_station', 'stop_id'])
stop_df_obb_mod = stop_df_obb_mod.drop_duplicates(subset=['stop_id'], keep='first')
stop_df_obb_mod['parent_station'] = stop_df_obb_mod['stop_id']
display(stop_df_obb_mod.head())

stop_times_df_obb_mod = stop_times_df_obb.copy()
stop_times_df_obb_mod['stop_id'] = stop_times_df_obb_mod['stop_id'].astype(str)
stop_times_df_obb_mod["stop_id"] = stop_times_df_obb_mod["stop_id"].apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)
display(stop_times_df_obb_mod.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:41:3087,Bad Sauerbrunn Bahnhof,47.773870,16.324886,NaN,NaN,at:41:3087,Level 0,1.0
6,at:41:3145,Baumgarten-Schattendorf Bahnhof,47.727514,16.518257,NaN,NaN,at:41:3145,Level 0,1.0
10,at:41:3186,Breitenbrunn Bahnhof,47.941229,16.746385,NaN,NaN,at:41:3186,Level 0,1.0
13,at:41:3294,Deutschkreutz Bahnhof,47.603827,16.620054,NaN,NaN,at:41:3294,Level 0,1.0
18,at:41:3298,Donnerskirchen Bahnhof,47.888571,16.660416,NaN,NaN,at:41:3298,Level 0,1.0


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,start_pickup_drop_off_window,end_pickup_drop_off_window,pickup_booking_rule_id,drop_off_booking_rule_id,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
0,1.TA.1-2-j24-1.1.H,00:02:00,00:02:00,at:49:334,1,NaN,NaN,NaN,NaN,NaN,0,0,0.00
1,1.TA.1-2-j24-1.1.H,00:05:00,00:05:00,at:49:1248,2,NaN,NaN,NaN,NaN,NaN,0,0,2164.70
2,1.TA.1-2-j24-1.1.H,00:08:00,00:09:00,at:49:769,3,NaN,NaN,NaN,NaN,NaN,0,0,4486.56
3,1.TA.1-2-j24-1.1.H,00:11:00,00:12:00,at:49:1357,4,NaN,NaN,NaN,NaN,NaN,0,0,7184.07
4,1.TA.1-2-j24-1.1.H,00:17:00,00:17:00,at:43:3279,5,NaN,NaN,NaN,NaN,NaN,0,0,13505.64


In [510]:
# remove from stop_times_df entries outside of the time window (6am-8pm)
stop_times_df_obb_mod = stop_times_df_obb_mod[stop_times_df_obb_mod['departure_time'].between('06:00:00', '20:00:00')]

stop_times_df_obb_mod.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,start_pickup_drop_off_window,end_pickup_drop_off_window,pickup_booking_rule_id,drop_off_booking_rule_id,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
20,1.TA.1-MB4-j24-1.1.R,13:34:00,13:34:00,at:48:134,1,NaN,NaN,NaN,NaN,NaN,0,0,0.00
21,1.TA.1-MB4-j24-1.1.R,13:35:00,13:35:00,at:48:892,2,NaN,NaN,NaN,NaN,NaN,0,0,997.51
22,1.TA.1-MB4-j24-1.1.R,13:37:00,13:37:00,at:48:139,3,NaN,NaN,NaN,NaN,NaN,0,0,2443.64
23,1.TA.1-MB4-j24-1.1.R,13:40:00,13:40:00,at:48:326,4,NaN,NaN,NaN,NaN,NaN,0,0,4566.15
24,1.TA.1-MB4-j24-1.1.R,13:43:00,13:44:00,at:48:187,5,NaN,NaN,NaN,NaN,NaN,0,0,5832.83


In [511]:
# remove all services entries in calendar_df that are not relevant for the specified day

calendar_df_obb_filtered = calendar_df_obb[(calendar_df_obb['start_date'] <= selected_day) & (calendar_df_obb['end_date'] >= selected_day)]
# TODO: check exceptions from calendar_dates_df
calendar_df_obb_filtered.head()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,MoFr,1,1,1,1,1,0,0,20231210,20241214
1,SaSo,0,0,0,0,0,1,1,20231210,20241214
2,TA,1,1,1,1,1,1,1,20231210,20241214
3,TA+00600,1,1,1,1,1,0,0,20231210,20241214
4,TA+01300,0,0,1,1,0,0,0,20231210,20241214


In [512]:
calendar_dates_df_obb_filtered = calendar_dates_df_obb[calendar_dates_df_obb["date"] == selected_day]
print(calendar_dates_df_obb_filtered.shape)
calendar_dates_df_obb_filtered.head()

(1011, 3)


,service_id,date,exception_type
110,MoFr,20240530,2
259,SaSo,20240530,1
286,TA+00600,20240530,2
394,TA+01300,20240530,2
592,TA+02000,20240530,2


In [513]:
# remove entries from trips_df that are not valid i.e only valid ones
trips_df_obb_filtered = trips_df_obb[trips_df_obb['service_id'].isin(calendar_df_obb_filtered['service_id'])]
print(trips_df_obb_filtered.shape)
trips_df_obb_filtered.head()

(20443, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,10-A10-j24-1,TA+a3,1.TA.10-A10-j24-1.1.R,10-A10-j24-1.1.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
1,10-A10-j24-1,TA+a2300,2.TA.10-A10-j24-1.2.R,10-A10-j24-1.2.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
2,10-A10-j24-1,TA+o6300,3.TA.10-A10-j24-1.3.R,10-A10-j24-1.3.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
3,10-A10-j24-1,TA+qf300,4.TA.10-A10-j24-1.4.R,10-A10-j24-1.4.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
4,10-A10-j24-1,TA+vW,5.TA.10-A10-j24-1.5.R,10-A10-j24-1.5.R,Wien Hauptbahnhof,D 14317,1.0,NaN


In [514]:
# remove all services that have exception_type == 2 (removed) in calendar_dates
trips_df_obb_filtered = trips_df_obb_filtered[trips_df_obb_filtered["service_id"].isin(calendar_dates_df_obb_filtered[calendar_dates_df_obb_filtered["exception_type"] == 2]["service_id"])]
print(trips_df_obb_filtered.shape)
trips_df_obb_filtered.head()

(5836, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
10,10-A11-j24-1,TA+kP,13.TA.10-A11-j24-1.7.R,10-A11-j24-1.7.R,Flughafen Wien Bahnhof,RJX 565,1.0,NaN
12,10-A11-j24-1,TA+22100,15.TA.10-A11-j24-1.7.R,10-A11-j24-1.7.R,Flughafen Wien Bahnhof,RJX 567,1.0,NaN
15,10-A11-j24-1,TA+ot500,18.TA.10-A11-j24-1.8.H,10-A11-j24-1.8.H,Zürich HB,RJX 162,0.0,1397.0
16,10-A11-j24-1,TA+oi,19.TA.10-A11-j24-1.8.H,10-A11-j24-1.8.H,Zürich HB,RJX 162,0.0,1398.0
20,10-A11-j24-1,TA+e9600,22.TA.10-A11-j24-1.8.H,10-A11-j24-1.8.H,Zürich HB,RJX 162,0.0,1399.0


In [515]:
# add  back services added in calendar_dates (exception_type=1)
# trips_filtered = trips_filtered + trips[service_id = calender_dates[exception_type=1][service_id]]
trips_df_obb_filtered_full = pd.concat([trips_df_obb_filtered, trips_df_obb[trips_df_obb["service_id"].isin(calendar_dates_df_obb_filtered[calendar_dates_df_obb_filtered["exception_type"]==1]["service_id"])]])
print(trips_df_obb_filtered_full.shape)
trips_df_obb_filtered_full.head()
#len(trips_full["trip_id"].unique())

(7651, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
10,10-A11-j24-1,TA+kP,13.TA.10-A11-j24-1.7.R,10-A11-j24-1.7.R,Flughafen Wien Bahnhof,RJX 565,1.0,NaN
12,10-A11-j24-1,TA+22100,15.TA.10-A11-j24-1.7.R,10-A11-j24-1.7.R,Flughafen Wien Bahnhof,RJX 567,1.0,NaN
15,10-A11-j24-1,TA+ot500,18.TA.10-A11-j24-1.8.H,10-A11-j24-1.8.H,Zürich HB,RJX 162,0.0,1397.0
16,10-A11-j24-1,TA+oi,19.TA.10-A11-j24-1.8.H,10-A11-j24-1.8.H,Zürich HB,RJX 162,0.0,1398.0
20,10-A11-j24-1,TA+e9600,22.TA.10-A11-j24-1.8.H,10-A11-j24-1.8.H,Zürich HB,RJX 162,0.0,1399.0


In [516]:
print(stop_times_df_obb_mod.shape)

# keep only valid trips from trips_df_obb_filtered_full
stop_times_df_obb_mod = stop_times_df_obb_mod[stop_times_df_obb_mod['trip_id'].isin(trips_df_obb_filtered_full['trip_id'])]

print(stop_times_df_obb_mod.shape)
stop_times_df_obb_mod.head()

(165634, 13)
(66583, 13)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,start_pickup_drop_off_window,end_pickup_drop_off_window,pickup_booking_rule_id,drop_off_booking_rule_id,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
81,1.TA.1-S3-K-j24-1.1.R,07:42:00,07:42:00,at:42:3842,1,NaN,NaN,NaN,NaN,NaN,0,0,0.00
82,1.TA.1-S3-K-j24-1.1.R,07:44:00,07:44:00,at:42:3642,2,NaN,NaN,NaN,NaN,NaN,0,0,1261.56
89,1.TA.1-S3-O-j24-1.1.R,07:15:00,07:15:00,at:44:41164,1,NaN,NaN,NaN,NaN,NaN,0,0,0.00
90,1.TA.1-S3-O-j24-1.1.R,07:19:00,07:19:00,at:44:41060,2,NaN,NaN,NaN,NaN,NaN,0,0,3194.77
91,1.TA.1-S3-S-j24-1.1.R,18:15:00,18:15:00,at:45:54001,1,NaN,NaN,NaN,NaN,NaN,0,0,0.00


In [517]:
df_tmp = stop_df_obb_mod.merge(stop_times_df_obb_mod, on='stop_id')
result_obb = df_tmp.groupby("parent_station").size().reset_index(name="count")
result_obb.rename(columns={"parent_station": "stop_id"}, inplace=True)
result_obb = result_obb[result_obb["stop_id"].str.startswith('at')]
display(result_obb.sort_values(by="count", ascending=False))

,stop_id,count
998,at:49:1349,1362
986,at:49:1015,1145
989,at:49:1091,817
1007,at:49:1705,801
1011,at:49:334,783
...,...,...
794,at:46:6154,1
798,at:46:6310,1
478,at:44:41016,1
803,at:46:6635,1


In [518]:
# merge result with stops parent stations
stops_obb_final = stop_df_obb_mod.merge(result_obb, on='stop_id', how='right')
stops_obb_final = stops_obb_final[['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'count']]
# calculate the interval
#stops_obb_final["interval"] = stops_obb_final["count"].apply(lambda x: 840 / (x/2))
#display(stops_obb_final.sort_values(by="interval", ascending=True))

display(stops_obb_final)

,stop_id,stop_name,stop_lat,stop_lon,count
0,at:41:3087,Bad Sauerbrunn Bahnhof,47.773870,16.324886,46
1,at:41:3145,Baumgarten-Schattendorf Bahnhof,47.727514,16.518257,39
2,at:41:3186,Breitenbrunn Bahnhof,47.941229,16.746385,41
3,at:41:3294,Deutschkreutz Bahnhof,47.603827,16.620054,46
4,at:41:3298,Donnerskirchen Bahnhof,47.888571,16.660416,41
...,...,...,...,...,...
1029,at:49:910,Wien Aspern Nord,48.233852,16.504630,61
1030,at:49:94,Wien Atzgersdorf,48.147016,16.288657,276
1031,at:49:948,Wien Nußdorf,48.260209,16.367664,50
1032,at:49:959,Wien Oberdöbling,48.244011,16.344101,31


In [519]:
stops_obb_final.to_csv(output_path + "stop_count_obb.csv", index=False)

### calc route types for oebb

In [520]:
print(routes_df_obb.shape)
print(trips_df_obb.shape)

route_type_obb = routes_df_obb.merge(trips_df_obb, on='route_id')
print(route_type_obb.shape)
route_type_obb.head()

(308, 5)
(20443, 8)
(20443, 12)


,route_id,agency_id,route_short_name,route_long_name,route_type,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,10-A10-j24-1,1,A10,NaN,2,TA+a3,1.TA.10-A10-j24-1.1.R,10-A10-j24-1.1.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
1,10-A10-j24-1,1,A10,NaN,2,TA+a2300,2.TA.10-A10-j24-1.2.R,10-A10-j24-1.2.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
2,10-A10-j24-1,1,A10,NaN,2,TA+o6300,3.TA.10-A10-j24-1.3.R,10-A10-j24-1.3.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
3,10-A10-j24-1,1,A10,NaN,2,TA+qf300,4.TA.10-A10-j24-1.4.R,10-A10-j24-1.4.R,Villach Hauptbahnhof,RJ 639,1.0,NaN
4,10-A10-j24-1,1,A10,NaN,2,TA+vW,5.TA.10-A10-j24-1.5.R,10-A10-j24-1.5.R,Wien Hauptbahnhof,D 14317,1.0,NaN


In [521]:
# transport_category = ["Fernverkehr REX", 
                    #   "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
                    #   "Straßenbahn, Metrobus, 0-Bus", 
                    #   "Bus"]

# {0: "Tram", 1: "Metro", 2: "Fernverkehr REX", 3: "Bus", 11: "Electric Bus", }
#route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3, }

def detect_obb_route_type(trips_name, route_type):
    if route_type == 2:
        trips_name = trips_name.lower()
        if any(x in trips_name for x in ["rj", "rjx", "nj", "en", "ic", "ec", "ice", "ecb", "rex"]): #fernverkehr
            return 0
        else:
            return 1
    else:
        return route_type_translation[route_type]
    # if any(x in trips_name for x in ["s", ]): #sbahn, regionalbahn, lokalbahn
    #     return 1


print(detect_obb_route_type("s 45", 3))

3


In [522]:
obb_station_ranks = trips_df_obb[["route_id", "trip_id", "service_id", "trip_short_name"]].merge(routes_df_obb[["route_id", "route_type"]], on='route_id')
display(obb_station_ranks.head())
obb_station_ranks = obb_station_ranks.merge(stop_times_df_obb_mod[["trip_id", "stop_id"]], on='trip_id')
display(obb_station_ranks.head())
obb_station_ranks = obb_station_ranks.merge(stop_df_obb_mod[["stop_id", "stop_name"]], on='stop_id')
display(obb_station_ranks.head())

,route_id,trip_id,service_id,trip_short_name,route_type
0,10-A10-j24-1,1.TA.10-A10-j24-1.1.R,TA+a3,RJ 639,2
1,10-A10-j24-1,2.TA.10-A10-j24-1.2.R,TA+a2300,RJ 639,2
2,10-A10-j24-1,3.TA.10-A10-j24-1.3.R,TA+o6300,RJ 639,2
3,10-A10-j24-1,4.TA.10-A10-j24-1.4.R,TA+qf300,RJ 639,2
4,10-A10-j24-1,5.TA.10-A10-j24-1.5.R,TA+vW,D 14317,2


,route_id,trip_id,service_id,trip_short_name,route_type,stop_id
0,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:48:817
1,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:48:130
2,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1222
3,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1212
4,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1204


,route_id,trip_id,service_id,trip_short_name,route_type,stop_id,stop_name
0,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:48:817,Feldkirch Bahnhof
1,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:48:130,Bludenz Bahnhof
2,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1222,St. Anton a. A. Bahnhof
3,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1212,Landeck-Zams Bahnhof
4,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1204,Ötztal-Bahnhof


In [523]:
obb_station_ranks = obb_station_ranks[obb_station_ranks['stop_id'].str.startswith('at')]

In [524]:
obb_station_ranks["rank"] = obb_station_ranks.apply(lambda x: detect_obb_route_type(x['trip_short_name'], x['route_type']), axis=1)
obb_station_ranks.head()

,route_id,trip_id,service_id,trip_short_name,route_type,stop_id,stop_name,rank
0,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:48:817,Feldkirch Bahnhof,0
1,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:48:130,Bludenz Bahnhof,0
2,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1222,St. Anton a. A. Bahnhof,0
3,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1212,Landeck-Zams Bahnhof,0
4,10-A11-j24-1,13.TA.10-A11-j24-1.7.R,TA+kP,RJX 565,2,at:47:1204,Ötztal-Bahnhof,0


In [525]:
obb_station_ranks = obb_station_ranks[['stop_id', 'stop_name', 'rank']]
print(obb_station_ranks.shape)

stop_rank_obb = obb_station_ranks.groupby(['stop_id', 'stop_name']).agg({'rank': 'min'}).reset_index()
print(stop_rank_obb.shape)
stop_rank_obb

(63366, 3)
(1034, 3)


,stop_id,stop_name,rank
0,at:41:3087,Bad Sauerbrunn Bahnhof,1
1,at:41:3145,Baumgarten-Schattendorf Bahnhof,0
2,at:41:3186,Breitenbrunn Bahnhof,0
3,at:41:3294,Deutschkreutz Bahnhof,0
4,at:41:3298,Donnerskirchen Bahnhof,0
...,...,...,...
1029,at:49:910,Wien Aspern Nord,0
1030,at:49:94,Wien Atzgersdorf,1
1031,at:49:948,Wien Nußdorf,1
1032,at:49:959,Wien Oberdöbling,1


In [526]:
stop_rank_obb.to_csv(output_path + "stop_rank_obb.csv", index=False)